# CUDA-Accelerated Image Dehazing on Google Colab

This notebook configures, builds, and runs the CUDA-accelerated Dark Channel Prior (DCP) dehazing pipeline on Google Colab with an NVIDIA GPU (e.g. Tesla T4).

### Pipeline Stages:
1. **Dark Channel Calculation**: Separable 2-pass min-filter with shared memory tiles (`dark_channel.cu`)
2. **Atmospheric Light Estimation**: Parallel histogram atomic binning + warp shuffle reduction (`atm_light.cu`)
3. **Transmission Map Estimation**: Normalized channel minimums with texture reads (`transmission.cu`)
4. **Transmission Refinement**: O(1) guided filter via Summed Area Tables (SAT) (`guided_filter.cu`)
5. **Radiance Recovery**: Coalesced pixel reconstruction with hardware saturation (`recovery.cu`)

## 1. Verify GPU Environment
Ensure a GPU (e.g. Tesla T4) is attached to the Colab session.

In [ ]:
# Verify NVIDIA GPU and CUDA driver
!nvidia-smi
!nvcc --version

## 2. Install Build Dependencies & Setup Workspace

In [ ]:
import os
from pathlib import Path

# Install build tools and OpenCV
!apt-get update -qq && apt-get install -y -qq cmake libopencv-dev ninja-build
!pip install -q pandas matplotlib numpy pillow tqdm

# If cloned from github in root
if not Path("CMakeLists.txt").exists():
    if Path("PCAP").exists():
        %cd PCAP
    else:
        !git clone https://github.com/aarav2912/PCAP.git
        %cd PCAP

print("Working directory:", os.getcwd())
!ls -la

## 3. Configure and Compile Targets (Architecture 75 for Tesla T4)

In [ ]:
# Configure CMake targeting Compute Capability 7.5 (Turing / Tesla T4)
!cmake -S . -B build \
    -DCMAKE_BUILD_TYPE=Release \
    -DCMAKE_CUDA_ARCHITECTURES=75

# Compile both dehazer and benchmark executables
!cmake --build build --parallel $(nproc)

# Verify binaries
!ls -lh build/dehazer build/benchmark

## 4. Single-Image Inference Smoke Test (Sample Dataset Mode)
Generate paired test images using the physical scattering model ($I = J \cdot t + A(1-t)$) for instant testing.

In [ ]:
# Generate synthetic sample paired images
!python data/download_reside.py --subset sample
!python data/download_reside.py --verify

sample_hazy = "data/sample/hazy/1400_1.png"
sample_gt   = "data/sample/gt/1400.png"

print("\n--- [1] CUDA Mode ---")
!./build/dehazer --mode cuda --input {sample_hazy} --output output_cuda.png --gt {sample_gt}

print("\n--- [2] CPU Sequential Mode ---")
!./build/dehazer --mode cpu --input {sample_hazy} --output output_cpu.png --gt {sample_gt}

print("\n--- [3] OpenMP Parallel Mode ---")
!./build/dehazer --mode omp --input {sample_hazy} --output output_omp.png --threads 4 --gt {sample_gt}

## 5. Visual Display: Input vs. Restored Output

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

fig, axes = plt.subplots(1, 4, figsize=(18, 5))
display_list = [
    ("Hazy Input", "data/sample/hazy/1400_1.png"),
    ("Ground Truth", "data/sample/gt/1400.png"),
    ("CUDA Restored", "output_cuda.png"),
    ("OpenMP Restored", "output_omp.png"),
]

for ax, (title, path) in zip(axes, display_list):
    if Path(path).exists():
        ax.imshow(Image.open(path))
    ax.set_title(title, fontsize=12, fontweight="bold")
    ax.axis("off")

plt.tight_layout()
plt.show()

## 6. (Optional) Full RESIDE SOTS Dataset Download
Download the full RESIDE SOTS test set using Kaggle CLI or a custom direct mirror.

In [ ]:
# Option A: Download via Kaggle CLI
# 1. Upload your kaggle.json to /content/
# 2. Run:
# !mkdir -p ~/.kaggle && cp /content/kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
# !python data/download_reside.py --subset sots --source kaggle

# Option B: Download via custom URL
# !python data/download_reside.py --subset sots --url <DIRECT_MIRROR_URL>

# Check current dataset state
!python data/download_reside.py --verify

## 7. 3-Way Batch Benchmark (CPU vs. OpenMP vs. CUDA) & Report Generation

In [ ]:
# Select dataset path (SOTS indoor if present, otherwise generated sample)
hazy_dir = "data/SOTS/indoor/hazy" if Path("data/SOTS/indoor/hazy").exists() else "data/sample/hazy"
gt_dir   = "data/SOTS/indoor/gt" if Path("data/SOTS/indoor/gt").exists() else "data/sample/gt"

print(f"Running benchmark with hazy_dir: {hazy_dir}, gt_dir: {gt_dir}")

# Execute 3-way benchmark
!./build/benchmark \
    --hazy {hazy_dir} \
    --gt {gt_dir} \
    --output results/ \
    --mode all \
    --threads 4

# Generate comparison charts and speedup summary
!python scripts/report.py --csv results/benchmark.csv --out results/

## 8. View Benchmark Speedup & Metric Graphs

In [ ]:
from IPython.display import Image as IPyImage, display

if Path("results/speedup_report.txt").exists():
    print(Path("results/speedup_report.txt").read_text())

for img_file in ["timing_comparison.png", "quality_metrics.png", "visual_comparison.png"]:
    p = Path("results") / img_file
    if p.exists():
        print(f"Displaying {img_file}:")
        display(IPyImage(str(p)))
